# 4.3 归一化与 FFN

这一节开始把前面学过的 reduction、广播、matmul 和激活函数组合成更接近模型结构的模块：**归一化层** 和 **FFN 模块**。

归一化层帮助稳定每一行 hidden state 的数值尺度；FFN 则把输入先投影到更大的中间维度，经过激活函数后再投影回来。读这两类模块时，不要先被代码长度吸引，先抓住公式、shape 和写回位置。

## 1. 环境准备

和前面的章节一样，先清理 Notebook 状态，再准备当前运行模式。

In [ ]:
import os
import math
from dataclasses import dataclass
from typing import Literal

import torch
import pypto
import numpy as np
from numpy.testing import assert_allclose

try:
    import torch_npu
except ImportError:
    torch_npu = None


def reset_pypto_notebook_state():
    try:
        pypto.reset()
    except Exception:
        pass

    try:
        from pypto._controller import Controller
        Controller.end_function()
    except Exception:
        pass


def get_device():
    if torch_npu is None:
        print("torch_npu is not available; the notebook will stay in SIM/CPU mode.")
        return "cpu"

    device_id = int(os.environ.get("TILE_FWK_DEVICE_ID", "0"))
    torch.npu.set_device(device_id)
    return f"npu:{device_id}"


reset_pypto_notebook_state()
device = get_device()
RUN_MODE = pypto.RunMode.NPU if device != "cpu" else pypto.RunMode.SIM

print("TILE_FWK_DEVICE_ID:", os.environ.get("TILE_FWK_DEVICE_ID", "<not set>"))
print("device:", device)
print("run_mode:", RUN_MODE)
print("pypto:", pypto.__file__)


## 2. 学完后你应该能够

学完这一节后，可以回到这里检查自己是否已经做到：

1. 说清楚 LayerNorm 和 RMSNorm 的计算公式与差别。
2. 看懂为什么归一化类算子通常要围绕 `sum`、`sqrt` 和广播来写。
3. 理解一个 FFN 模块是如何组合出不同激活分支的。
4. 看懂静态 FFN 与动态 FFN 的写法差异。
5. 用 PyTorch reference 验证自己的 PyPTO 结果。

## 3. 这一节会依次练什么

| 例子 | 主题 | 关键点 |
| --- | --- | --- |
| LayerNorm | 标准归一化 | `mean`、`var`、`gamma`、`beta` |
| RMSNorm | 简化归一化 | `sqrt(mean(x^2) + eps)` |
| FFN + GELU | 标准前馈网络 | `matmul -> activation -> matmul` |
| FFN + SwiGLU | 门控前馈网络 | 两路投影与门控组合 |
| FFN + ReLU | 简化激活分支 | 基础激活替换 |
| Dynamic FFN + GELU | 动态 batch | `view`、`loop`、`assemble` |

这些练习的共同点是：都在把多个基础算子组织成更大的模块，而不是单独使用某一个 API。

### 3.1 建议运行顺序

建议按下面顺序阅读和运行。先理解归一化里的 reduction 与广播，再进入 FFN 里的 matmul 与激活函数组合：

| 步骤 | 你会看到什么 | 为什么先看它 |
| --- | --- | --- |
| LayerNorm | `mean -> var -> normalize -> scale + bias` | 先看完整归一化公式。 |
| RMSNorm | `mean square -> rms -> scale` | 和 LayerNorm 对比，理解省掉均值路径后的变化。 |
| 静态 FFN | `matmul -> activation -> matmul` | 先掌握固定 shape 下最直接的模块组合。 |
| 门控 FFN | `gate` 与 `up` 两路投影 | 观察 SwiGLU 这类门控结构如何改变中间分支。 |
| 动态 FFN | `loop -> view -> compute -> assemble` | 最后再看 batch 变化时如何分块处理。 |

如果正在 Notebook 中跟着运行，先执行环境准备单元，再执行当前模块的 kernel 定义单元，最后运行测试单元。修改过 JIT kernel 后，建议重新运行环境准备单元和当前 kernel 定义单元，避免旧的记录状态影响结果。

## 4. LayerNorm 与 RMSNorm

归一化层的目标很直接：让每一行或每一组特征保持稳定的数值分布。这样后续的线性层和激活函数更容易工作，训练和推理也更稳。

LayerNorm 和 RMSNorm 的差别在于：LayerNorm 会同时处理均值和方差，RMSNorm 则只保留均方根这条路径，所以更轻一些。

### 4.1 先从公式看 shape

这里先使用二维 shape 观察归一化：

```text
x:      [batch_size, hidden_size] = [32, 128]
gamma:  [hidden_size] = [128]
beta:   [hidden_size] = [128]
output: [batch_size, hidden_size] = [32, 128]
```

LayerNorm 沿最后一维计算：

```text
mean = sum(x, dim=-1, keepdim=True) / hidden_size
centered = x - mean
var = sum(centered * centered, dim=-1, keepdim=True) / hidden_size
normalized = centered / sqrt(var + eps)
output = normalized * gamma + beta
```

RMSNorm 没有减均值，也没有 `beta`：

```text
mean_sq = sum(x * x, dim=-1, keepdim=True) / hidden_size
rms = sqrt(mean_sq + eps)
output = (x / rms) * gamma
```

`keepdim=True` 会把 `mean`、`var`、`rms` 保持成 `[32, 1]`，这样它们可以自然广播回 `[32, 128]`。`gamma` 和 `beta` 是 `[128]`，也会沿 batch 维广播到每一行。

In [ ]:
@dataclass
class NormConfig:
    norm_type: Literal["layernorm", "rmsnorm"] = "layernorm"
    eps: float = 1e-6
    dtype: pypto.DataType = pypto.DT_BF16
    use_dynamic_shape: bool = False


def layernorm_golden(x: torch.Tensor, gamma: torch.Tensor, beta: torch.Tensor, eps: float) -> torch.Tensor:
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False)
    normalized = (x - mean) / torch.sqrt(var + eps)
    return normalized * gamma + beta


def layernorm_core(x: pypto.Tensor, gamma: pypto.Tensor, beta: pypto.Tensor,
                   eps: float, hidden_size: int) -> pypto.Tensor:
    mean = pypto.sum(x, dim=-1, keepdim=True)
    mean = mean / hidden_size
    centered = x - mean
    squared = centered * centered
    var = pypto.sum(squared, dim=-1, keepdim=True)
    var = var / hidden_size
    std = pypto.sqrt(var + eps)
    normalized = centered / std
    scaled = normalized * gamma
    return scaled + beta


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def layer_norm_kernel(
    x: pypto.Tensor(),
    gamma: pypto.Tensor(),
    beta: pypto.Tensor(),
    output: pypto.Tensor(),
    config: NormConfig):
    hidden_size = x.shape[1]
    pypto.set_vec_tile_shapes(64, 128)
    out = layernorm_core(x, gamma, beta, config.eps, hidden_size)
    pypto.assemble(out, [0, 0], output)


def rmsnorm_golden(x: torch.Tensor, gamma: torch.Tensor, eps: float) -> torch.Tensor:
    rms = torch.sqrt((x ** 2).mean(dim=-1, keepdim=True) + eps)
    return (x / rms) * gamma


def rms_norm_core(x: pypto.Tensor, gamma: pypto.Tensor, eps: float, hidden_size: int) -> pypto.Tensor:
    squared = x * x
    mean_sq = pypto.sum(squared, dim=-1, keepdim=True)
    mean_sq = mean_sq / hidden_size
    rms = pypto.sqrt(mean_sq + eps)
    normalized = x / rms
    return normalized * gamma


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def rms_norm_kernel(
    x: pypto.Tensor(),
    gamma: pypto.Tensor(),
    output: pypto.Tensor(),
    config: NormConfig):
    hidden_size = x.shape[1]
    pypto.set_vec_tile_shapes(64, 128)
    out = rms_norm_core(x, gamma, config.eps, hidden_size)
    pypto.assemble(out, [0, 0], output)


### 4.2 `NormConfig` 与归一化 kernel 逐段说明

`NormConfig` 是归一化 kernel 的配置对象。它不承载真实输入数据，而是把运行时需要的少量参数放在一起传入 kernel。

| 字段 | 常用取值 | 含义 |
| --- | --- | --- |
| `norm_type` | `"layernorm"` 或 `"rmsnorm"` | 标记当前测试是哪类归一化。这里两个 kernel 分开定义，所以它主要用于可读性。 |
| `eps` | `1e-6` | 加到方差或均方根上，避免除以 0。 |
| `dtype` | `pypto.DT_BF16` | 说明输出和主要输入使用 BF16。 |
| `use_dynamic_shape` | `False` | 是否使用动态 shape。这里的归一化练习先使用静态二维输入。 |

LayerNorm 的核心函数和公式几乎一一对应：

| 代码 | 输入 shape | 输出 shape | 含义 |
| --- | --- | --- | --- |
| `mean = pypto.sum(x, dim=-1, keepdim=True)` | `[B, H]` | `[B, 1]` | 每一行求和。 |
| `mean = mean / hidden_size` | `[B, 1]` | `[B, 1]` | 求每一行均值。 |
| `centered = x - mean` | `[B, H]` 和 `[B, 1]` | `[B, H]` | 均值通过广播减回每个元素。 |
| `squared = centered * centered` | `[B, H]` | `[B, H]` | 为方差准备平方项。 |
| `var = pypto.sum(squared, dim=-1, keepdim=True)` | `[B, H]` | `[B, 1]` | 每一行平方和。 |
| `var = var / hidden_size` | `[B, 1]` | `[B, 1]` | 得到方差。 |
| `std = pypto.sqrt(var + eps)` | `[B, 1]` | `[B, 1]` | 得到标准差，`eps` 防止除 0。 |
| `normalized = centered / std` | `[B, H]` 和 `[B, 1]` | `[B, H]` | 归一化到稳定尺度。 |
| `scaled = normalized * gamma` | `[B, H]` 和 `[H]` | `[B, H]` | 使用可学习缩放参数。 |
| `return scaled + beta` | `[B, H]` 和 `[H]` | `[B, H]` | 使用可学习平移参数。 |

`layer_norm_kernel` 外层只做三件事：从 `x.shape[1]` 取 `hidden_size`，设置 vec tile，然后调用 `layernorm_core` 并用 `pypto.assemble(out, [0, 0], output)` 写回完整输出。这里虽然没有切块，但仍使用 `assemble`，和后面的动态 FFN 保持一致的写回风格。

### 4.3 RMSNorm 逐段说明

RMSNorm 可以看作 LayerNorm 的简化版本：不减均值，不使用 `beta`，只用均方根控制尺度。

| 代码 | 输入 shape | 输出 shape | 含义 |
| --- | --- | --- | --- |
| `squared = x * x` | `[B, H]` | `[B, H]` | 逐元素平方。 |
| `mean_sq = pypto.sum(squared, dim=-1, keepdim=True)` | `[B, H]` | `[B, 1]` | 每一行平方和。 |
| `mean_sq = mean_sq / hidden_size` | `[B, 1]` | `[B, 1]` | 得到平方均值。 |
| `rms = pypto.sqrt(mean_sq + eps)` | `[B, 1]` | `[B, 1]` | 得到均方根。 |
| `normalized = x / rms` | `[B, H]` 和 `[B, 1]` | `[B, H]` | 按每一行的 rms 归一化。 |
| `return normalized * gamma` | `[B, H]` 和 `[H]` | `[B, H]` | 使用可学习缩放参数。 |

对比 LayerNorm，可以看到 RMSNorm 少了 `mean`、`centered`、`var` 和 `beta` 相关路径。它不是“更粗糙的 LayerNorm”，而是另一种常见设计取舍：减少计算和参数，同时保留尺度归一化能力。

In [ ]:
def test_layer_norm(device_id=None, dynamic: bool = False):
    device_local = f'npu:{device_id}' if (device_id is not None and RUN_MODE == pypto.RunMode.NPU) else device
    batch_size, hidden_size = 32, 128
    shape = (batch_size, hidden_size)
    x_torch = torch.randn(shape, dtype=torch.bfloat16, device=device_local)
    gamma_torch = torch.ones(hidden_size, dtype=torch.bfloat16, device=device_local)
    beta_torch = torch.zeros(hidden_size, dtype=torch.bfloat16, device=device_local)
    config = NormConfig(norm_type="layernorm", dtype=pypto.DT_BF16)
    out_torch = torch.empty(shape, dtype=torch.bfloat16, device=device_local)
    layer_norm_kernel(x_torch, gamma_torch, beta_torch, out_torch, config)
    expected = layernorm_golden(x_torch, gamma_torch, beta_torch, config.eps)
    max_diff = (out_torch - expected).abs().max().item()
    print(f"LayerNorm input shape: {x_torch.shape}")
    print(f"LayerNorm output shape: {out_torch.shape}")
    print(f"LayerNorm max difference: {max_diff:.6f}")
    if RUN_MODE == pypto.RunMode.NPU:
        assert max_diff < 1e-1, "LayerNorm result mismatch!"


def test_rms_norm(device_id=None, dynamic: bool = False) -> None:
    device_local = f'npu:{device_id}' if (device_id is not None and RUN_MODE == pypto.RunMode.NPU) else device
    batch_size, hidden_size = 32, 128
    shape = (batch_size, hidden_size)
    x_torch = torch.randn(shape, dtype=torch.bfloat16, device=device_local)
    gamma_torch = torch.ones(hidden_size, dtype=torch.bfloat16, device=device_local)
    config = NormConfig(norm_type="rmsnorm", dtype=pypto.DT_BF16)
    out_torch = torch.empty(shape, dtype=torch.bfloat16, device=device_local)
    rms_norm_kernel(x_torch, gamma_torch, out_torch, config)
    expected = rmsnorm_golden(x_torch, gamma_torch, config.eps)
    max_diff = (out_torch - expected).abs().max().item()
    print(f"RMSNorm input shape: {x_torch.shape}")
    print(f"RMSNorm output shape: {out_torch.shape}")
    print(f"RMSNorm max difference: {max_diff:.6f}")
    if RUN_MODE == pypto.RunMode.NPU:
        assert max_diff < 1e-1, "RMSNorm result mismatch!"


### 4.4 归一化验证逻辑

LayerNorm 和 RMSNorm 的验证方式很像，都是先在 host 侧构造输入，再把 PyTorch reference 算出来，对比 PyPTO 输出。

| 代码 | 作用 |
| --- | --- |
| `shape = (batch_size, hidden_size)` | 归一化沿最后一维做，所以用二维输入最直观。 |
| `x_torch = torch.randn(..., dtype=torch.bfloat16)` | 构造真实 BF16 输入，用来观察归一化在低精度下的误差。 |
| `gamma_torch = torch.ones(hidden_size, ...)` | 初始缩放为 1，不改变归一化结果尺度。 |
| `beta_torch = torch.zeros(hidden_size, ...)` | LayerNorm 初始平移为 0。RMSNorm 不使用 `beta`。 |
| `out_torch = torch.empty(shape, ...)` | 由 PyPTO kernel 写入完整输出。 |
| `layernorm_golden(...)` / `rmsnorm_golden(...)` | 用 PyTorch 按同一公式计算 reference。 |
| `max_diff = ...abs().max().item()` | 对整个输出 Tensor 做最大误差检查。 |

差别主要在算子链：LayerNorm 需要均值和方差，RMSNorm 只需要均方根，所以后者更短一点。BF16 场景下误差会比 FP32 更明显，这里使用 `1e-1` 作为较宽松的判断阈值。

In [ ]:
test_layer_norm()
test_rms_norm()


## 5. FFN 模块

FFN 是 Transformer 里最基础也最核心的网络模块之一。它的标准结构很简单：

`Input -> Up Projection -> Activation -> Down Projection -> Output`

下面会依次看到 GELU、SwiGLU 和 ReLU 三种写法，并额外练习一个动态 batch 的 GELU 版本。

In [ ]:
@dataclass
class FFNConfig:
    batch_size: int
    hidden_size: int
    intermediate_size: int
    activation: Literal["gelu", "swiglu", "relu"] = "gelu"
    dtype: pypto.DataType = pypto.DT_FP16
    use_dynamic_shape: bool = False
    vec_tile_shape: tuple = (64, 128)
    cube_tile_shape: tuple = (64, 128, 128)
    basic_batch: int = 32


F_1 = 1.0
F_NEGA_1 = -1.0
GELU_COEFF = 1.702


def gelu_torch(x):
    return x * torch.sigmoid(GELU_COEFF * x)


def swiglu_torch(gate, up):
    swish = gate * torch.sigmoid(gate)
    return swish * up


def ceil_div(a, b):
    return (a + b - 1) // b


def relu_activation_core(x: pypto.tensor) -> pypto.tensor:
    pypto.set_vec_tile_shapes(*x.shape[:2] if len(x.shape) >= 2 else (32, 128))
    zero = pypto.full(x.shape, 0, x.dtype, valid_shape=x.shape)
    return pypto.maximum(x, zero)


def gelu_activation_core(x: pypto.tensor) -> pypto.tensor:
    pypto.set_vec_tile_shapes(*x.shape[:2] if len(x.shape) >= 2 else (32, 128))
    x_fp32 = pypto.cast(x, pypto.DT_FP32)
    x_scaled = pypto.mul(x_fp32, GELU_COEFF)
    x_neg = pypto.mul(x_scaled, F_NEGA_1)
    exp_neg = pypto.exp(x_neg)
    ones = pypto.full(exp_neg.shape, 1.0, exp_neg.dtype, valid_shape=exp_neg.shape)
    sigmoid = pypto.div(ones, pypto.add(exp_neg, F_1))
    activated = pypto.cast(pypto.mul(x_fp32, sigmoid), pypto.DT_BF16)
    return activated


def swiglu_activation_core(gate: pypto.tensor, up: pypto.tensor) -> pypto.tensor:
    gate_fp32 = pypto.cast(gate, pypto.DT_FP32)
    up_fp32 = pypto.cast(up, pypto.DT_FP32)
    pypto.set_vec_tile_shapes(*gate.shape[:2] if len(gate.shape) >= 2 else (32, 128))
    gate_neg = pypto.mul(gate_fp32, F_NEGA_1)
    exp_neg = pypto.exp(gate_neg)
    ones = pypto.full(exp_neg.shape, F_1, exp_neg.dtype, valid_shape=exp_neg.shape)
    sigmoid = pypto.div(ones, pypto.add(exp_neg, ones))
    swish = pypto.mul(gate_fp32, sigmoid)
    return pypto.cast(pypto.mul(swish, up_fp32), pypto.DT_BF16)


### 5.1 `FFNConfig` 与 FFN shape 总览

FFN 的核心不是某一个特别复杂的 API，而是矩阵乘法和激活函数的串联。你可以把它理解成一条很规整的流水线：

- 先把 hidden state 投影到中间维度。
- 再在中间维度上做激活。
- 最后投影回输出维度。

SwiGLU 比标准 GELU 多了一路 gate 投影，所以更像是“门控版 FFN”。

`FFNConfig` 把 shape、激活类型、dtype 和 tiling 参数集中起来：

| 字段 | 含义 |
| --- | --- |
| `batch_size` | 输入 batch 大小。静态测试中通常是 16，动态测试中是 32。 |
| `hidden_size` | 输入和输出 hidden 维度。静态测试是 128，动态测试是 512。 |
| `intermediate_size` | FFN 中间维度，通常大于 hidden 维度。这里使用 1024。 |
| `activation` | 选择 `gelu`、`swiglu` 或 `relu` 分支。 |
| `dtype` | matmul 和输出使用的数据类型，如 `pypto.DT_BF16` 或 `pypto.DT_FP16`。 |
| `use_dynamic_shape` | 是否走动态 batch 版本。 |
| `vec_tile_shape` | 激活、逐元素计算使用的向量 tile。 |
| `cube_tile_shape` | matmul 使用的 cube tile。 |
| `basic_batch` | 动态 FFN 每次处理的 batch 块大小。 |

静态 GELU/ReLU 的 shape 流是：

```text
hidden_states: [B, H]
up_weight:     [H, I]
up:            [B, I]
activated:     [B, I]
down_weight:   [I, H]
output:        [B, H]
```

静态 SwiGLU 多一路 gate：

```text
gate = hidden_states @ gate_weight -> [B, I]
up   = hidden_states @ up_weight   -> [B, I]
activated = swiglu(gate, up)       -> [B, I]
output = activated @ down_weight   -> [B, H]
```

这一节的 FFN 例子本质上都在复用同一个主干：`matmul -> activation -> matmul`。区别只在激活分支的选择，以及动态版本是否要先把 batch 拆成块。

### 5.2 三个激活 core 逐段说明

FFN 中的激活 core 都作用在 `[B, I]` 的中间张量上，输出 shape 也保持 `[B, I]`。

| 函数 | 关键代码 | 含义 |
| --- | --- | --- |
| `relu_activation_core` | `zero = pypto.full(...)`、`pypto.maximum(x, zero)` | 构造同 shape 的 0 Tensor，再做逐元素最大值。 |
| `gelu_activation_core` | `cast -> mul -> exp -> div -> mul -> cast` | 先转 FP32 计算 sigmoid 近似 GELU，最后转回 BF16。 |
| `swiglu_activation_core` | `gate_fp32`、`up_fp32`、`swish * up_fp32` | 两路输入先转 FP32，再计算 Swish gate 并调制 up 分支。 |

`gelu_activation_core` 没有直接调用 `pypto.sigmoid`，而是手工拆成 `exp` 和 `div`：

```text
sigmoid(x) = 1 / (1 + exp(-x))
gelu_approx(x) = x * sigmoid(1.702 * x)
```

这样写能展示复杂激活函数如何由更基础的 elementwise API 组合出来，也和 4.2 的公式保持一致。

In [ ]:
@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def dynamic_gelu_activation_core(
    hidden_states: pypto.tensor(),
    up_proj_weight: pypto.tensor(),
    down_proj_weight: pypto.tensor(),
    output: pypto.tensor(),
    config: FFNConfig):
    pypto.set_cube_tile_shapes(
        [config.cube_tile_shape[0], config.cube_tile_shape[0]],
        [config.cube_tile_shape[1], config.cube_tile_shape[1]],
        [config.cube_tile_shape[2], config.cube_tile_shape[2]]
    )
    pypto.set_vec_tile_shapes(*config.vec_tile_shape)
    hidden_size, intermediate_size = config.hidden_size, config.intermediate_size
    basic_batch = config.basic_batch
    batch_size = hidden_states.shape[0]
    num_iterations = ceil_div(batch_size, basic_batch)
    for idx in pypto.loop(0, num_iterations, 1, name="LOOP_FFN_BATCH", idx_name="idx"):
        batch_offset = idx * basic_batch
        hidden_chunk = pypto.view(
            hidden_states,
            [basic_batch, hidden_size],
            [batch_offset, 0],
            valid_shape=[(batch_size - batch_offset).min(basic_batch), hidden_size]
        )
        up = pypto.matmul(hidden_chunk, up_proj_weight, config.dtype)
        pypto.set_vec_tile_shapes(*config.vec_tile_shape)
        activated = gelu_activation_core(up)
        pypto.set_cube_tile_shapes(
            [config.cube_tile_shape[0], config.cube_tile_shape[0]],
            [config.cube_tile_shape[1], config.cube_tile_shape[1]],
            [config.cube_tile_shape[2], config.cube_tile_shape[2]]
        )
        pypto.set_matrix_size([basic_batch, intermediate_size, hidden_size])
        output_chunk = pypto.matmul(activated, down_proj_weight, config.dtype, b_trans=False)
        pypto.assemble(output_chunk, [batch_offset, 0], output)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def ffn_activation_kernel(
    hidden_states: pypto.tensor(),
    gate_proj_weight: pypto.tensor(),
    up_proj_weight: pypto.tensor(),
    down_proj_weight: pypto.tensor(),
    output: pypto.tensor(),
    config: FFNConfig):
    pypto.set_cube_tile_shapes(
        [config.cube_tile_shape[0], config.cube_tile_shape[0]],
        [config.cube_tile_shape[1], config.cube_tile_shape[1]],
        [config.cube_tile_shape[2], config.cube_tile_shape[2]]
    )
    pypto.set_vec_tile_shapes(*config.vec_tile_shape)
    if config.activation == "gelu":
        up = pypto.matmul(hidden_states, up_proj_weight, config.dtype)
        activated = gelu_activation_core(up)
    elif config.activation == "swiglu":
        gate = pypto.matmul(hidden_states, gate_proj_weight, config.dtype)
        up = pypto.matmul(hidden_states, up_proj_weight, config.dtype)
        activated = swiglu_activation_core(gate, up)
    elif config.activation == "relu":
        up = pypto.matmul(hidden_states, up_proj_weight, config.dtype)
        activated = relu_activation_core(up)
    else:
        raise ValueError(f"Unsupported activation: {config.activation}")
    result = pypto.matmul(activated, down_proj_weight, config.dtype, b_trans=False)
    pypto.assemble(result, [0, 0], output)


### 5.3 静态 FFN kernel 逐段说明

静态 FFN 的特点是 batch 和 hidden 的形状都固定，写法最直接。`ffn_activation_kernel` 会根据 `config.activation` 选择不同的分支，因此一个 kernel 就能覆盖 GELU、SwiGLU 和 ReLU 三类写法。

| 代码 | 作用 |
| --- | --- |
| `pypto.set_cube_tile_shapes(...)` | 为后续 `matmul` 设置 cube tile。 |
| `pypto.set_vec_tile_shapes(...)` | 为激活函数里的逐元素计算设置 vec tile。 |
| `if config.activation == "gelu"` | 只计算 `up` 投影，再走 GELU 近似。 |
| `elif config.activation == "swiglu"` | 同时计算 `gate` 和 `up` 两路投影，再做门控激活。 |
| `elif config.activation == "relu"` | 只计算 `up` 投影，再用 ReLU 替换 GELU。 |
| `result = pypto.matmul(activated, down_proj_weight, ...)` | 把 `[B, I]` 投影回 `[B, H]`。 |
| `pypto.assemble(result, [0, 0], output)` | 将完整静态结果写回输出。 |

注意 `gate_proj_weight` 在 GELU 和 ReLU 分支中并不会参与实际计算。它仍然作为参数传入，是为了让一个统一 kernel 签名覆盖三种分支，避免为每个激活函数单独设计完全不同的入口。

从 shape 角度看，静态 FFN 的阅读顺序非常固定：先看输入 `[B, H]`，再看投影权重 `[H, I]` 和 `[I, H]`，最后看中间激活是否保持 `[B, I]` 不变。只要这个链条顺了，静态 FFN 就很好读。

In [ ]:
def test_ffn_static_gelu(device_id=None):
    device_local = f'npu:{device_id}' if (device_id is not None and RUN_MODE == pypto.RunMode.NPU) else device
    batch_size = 16
    hidden_size = 128
    intermediate_size = 1024
    dtype = torch.bfloat16
    config = FFNConfig(
        batch_size=batch_size,
        hidden_size=hidden_size,
        intermediate_size=intermediate_size,
        activation="gelu",
        dtype=pypto.DT_BF16,
        use_dynamic_shape=False,
        vec_tile_shape=(16, 32),
        cube_tile_shape=(16, 32, 32),
    )
    hidden_states_torch = torch.randn(batch_size, hidden_size, dtype=dtype, device=device_local) / math.sqrt(batch_size)
    gate_proj_weight_torch = torch.randn(hidden_size, intermediate_size, dtype=dtype, device=device_local) / math.sqrt(batch_size)
    up_proj_weight_torch = torch.randn(hidden_size, intermediate_size, dtype=dtype, device=device_local) / math.sqrt(batch_size)
    down_proj_weight_torch = torch.randn(intermediate_size, hidden_size, dtype=dtype, device=device_local) / math.sqrt(batch_size)
    up_torch = torch.matmul(hidden_states_torch, up_proj_weight_torch)
    up_activated_torch = gelu_torch(up_torch.float()).to(dtype)
    output_torch_ref = torch.matmul(up_activated_torch, down_proj_weight_torch)
    output = torch.empty(batch_size, hidden_size, dtype=dtype, device=device_local)
    ffn_activation_kernel(hidden_states_torch, gate_proj_weight_torch, up_proj_weight_torch,
                          down_proj_weight_torch, output, config)
    print(f"Static GELU input shape: {hidden_states_torch.shape}")
    print(f"Static GELU output shape: {output_torch_ref.shape}")
    if RUN_MODE == pypto.RunMode.NPU:
        assert_allclose(output.cpu().to(torch.float32), output_torch_ref.cpu().to(torch.float32), rtol=3e-3, atol=3e-3)


def test_ffn_static_swiglu(device_id=None):
    device_local = f'npu:{device_id}' if (device_id is not None and RUN_MODE == pypto.RunMode.NPU) else device
    batch_size = 16
    hidden_size = 128
    intermediate_size = 1024
    dtype = torch.bfloat16
    config = FFNConfig(
        batch_size=batch_size,
        hidden_size=hidden_size,
        intermediate_size=intermediate_size,
        activation="swiglu",
        dtype=pypto.DT_BF16,
        use_dynamic_shape=False,
        vec_tile_shape=(16, 32),
        cube_tile_shape=(16, 32, 32),
    )
    hidden_states_torch = torch.randn(batch_size, hidden_size, dtype=dtype, device=device_local) / math.sqrt(batch_size)
    gate_proj_weight_torch = torch.randn(hidden_size, intermediate_size, dtype=dtype, device=device_local) / math.sqrt(batch_size)
    up_proj_weight_torch = torch.randn(hidden_size, intermediate_size, dtype=dtype, device=device_local) / math.sqrt(batch_size)
    down_proj_weight_torch = torch.randn(intermediate_size, hidden_size, dtype=dtype, device=device_local) / math.sqrt(batch_size)
    gate_torch = torch.matmul(hidden_states_torch, gate_proj_weight_torch)
    up_torch = torch.matmul(hidden_states_torch, up_proj_weight_torch)
    activated_torch = swiglu_torch(gate_torch.float(), up_torch.float()).to(dtype)
    output_torch_ref = torch.matmul(activated_torch, down_proj_weight_torch)
    output = torch.empty(batch_size, hidden_size, dtype=dtype, device=device_local)
    ffn_activation_kernel(hidden_states_torch, gate_proj_weight_torch, up_proj_weight_torch,
                          down_proj_weight_torch, output, config)
    print(f"Static SwiGLU input shape: {hidden_states_torch.shape}")
    print(f"Static SwiGLU output shape: {output_torch_ref.shape}")
    if RUN_MODE == pypto.RunMode.NPU:
        assert_allclose(output.cpu().to(torch.float32), output_torch_ref.cpu().to(torch.float32), rtol=3e-3, atol=3e-3)


def test_ffn_static_relu(device_id: int = None):
    device_local = f'npu:{device_id}' if (device_id is not None and RUN_MODE == pypto.RunMode.NPU) else device
    batch_size = 16
    hidden_size = 128
    intermediate_size = 1024
    dtype = torch.float16
    config = FFNConfig(
        batch_size=batch_size,
        hidden_size=hidden_size,
        intermediate_size=intermediate_size,
        activation="relu",
        dtype=pypto.DT_FP16,
        use_dynamic_shape=False,
        vec_tile_shape=(32, 64),
        cube_tile_shape=(32, 64, 64),
    )
    hidden_states_torch = torch.randn(batch_size, hidden_size, dtype=dtype, device=device_local) / math.sqrt(batch_size)
    gate_proj_weight_torch = torch.randn(hidden_size, intermediate_size, dtype=dtype, device=device_local) / math.sqrt(batch_size)
    up_proj_weight_torch = torch.randn(hidden_size, intermediate_size, dtype=dtype, device=device_local) / math.sqrt(batch_size)
    down_proj_weight_torch = torch.randn(intermediate_size, hidden_size, dtype=dtype, device=device_local) / math.sqrt(batch_size)
    up_torch = torch.matmul(hidden_states_torch, up_proj_weight_torch)
    up_activated_torch = torch.relu(up_torch)
    output_torch_ref = torch.matmul(up_activated_torch, down_proj_weight_torch)
    output = torch.empty(batch_size, hidden_size, dtype=dtype, device=device_local)
    ffn_activation_kernel(hidden_states_torch, gate_proj_weight_torch, up_proj_weight_torch,
                          down_proj_weight_torch, output, config)
    max_diff = np.abs((output.cpu().numpy() - output_torch_ref.cpu().numpy())).max()
    print(f"Static ReLU input shape: {hidden_states_torch.shape}")
    print(f"Static ReLU output shape: {output_torch_ref.shape}")
    print(f"Static ReLU max difference: {max_diff:.6f}")
    if RUN_MODE == pypto.RunMode.NPU:
        assert_allclose(output.cpu().to(torch.float32), output_torch_ref.cpu().to(torch.float32), rtol=3e-3, atol=3e-3)


### 5.4 静态 FFN 验证逻辑

静态 FFN 这三组测试分别覆盖了 GELU、SwiGLU 和 ReLU。它们的共同结构是相同的，差异只在激活分支。

| 测试 | 输入 shape | 中间 shape | 输出 shape |
| --- | --- | --- | --- |
| `test_ffn_static_gelu()` | `[16, 128]` | `[16, 1024]` | `[16, 128]` |
| `test_ffn_static_swiglu()` | `[16, 128]` | `gate=[16, 1024]`, `up=[16, 1024]` | `[16, 128]` |
| `test_ffn_static_relu()` | `[16, 128]` | `[16, 1024]` | `[16, 128]` |

验证时先在 host 侧构造 `hidden_states_torch`、`gate_proj_weight_torch`、`up_proj_weight_torch`、`down_proj_weight_torch`，再用 PyTorch 写同样的参考链路：

```text
hidden_states @ up_weight -> up
up -> activation -> activated
activated @ down_weight -> output
```

这就是为什么把激活函数单独提出来学很重要：一旦你认得了分支变化，FFN 模块其实就非常规整了。

In [ ]:
test_ffn_static_gelu()
test_ffn_static_swiglu()
test_ffn_static_relu()


### 5.5 动态 FFN

动态 FFN 的重点是 batch size 不再固定。这里的练习只对 GELU 分支做动态处理，但它已经把动态场景最常见的模式展示得很完整：`view` 切块、`loop` 迭代、最后用 `assemble` 写回输出。

In [ ]:
def test_ffn_dynamic_gelu(device_id: int = None, dynamic: bool = True):
    device_local = f'npu:{device_id}' if (device_id is not None and RUN_MODE == pypto.RunMode.NPU) else device
    batch_size = 32
    hidden_size = 512
    intermediate_size = 1024
    basic_batch = 16
    dtype = torch.bfloat16
    config = FFNConfig(
        batch_size=batch_size,
        hidden_size=hidden_size,
        intermediate_size=intermediate_size,
        activation="gelu",
        dtype=pypto.DT_BF16,
        use_dynamic_shape=dynamic,
        vec_tile_shape=(32, 64),
        cube_tile_shape=(32, 64, 64),
        basic_batch=basic_batch,
    )
    hidden_states_torch = torch.randn(batch_size, hidden_size, dtype=dtype, device=device_local) / math.sqrt(batch_size)
    gate_proj_weight_torch = torch.randn(hidden_size, intermediate_size, dtype=dtype, device=device_local) / math.sqrt(batch_size)
    up_proj_weight_torch = torch.randn(hidden_size, intermediate_size, dtype=dtype, device=device_local) / math.sqrt(batch_size)
    down_proj_weight_torch = torch.randn(intermediate_size, hidden_size, dtype=dtype, device=device_local) / math.sqrt(batch_size)
    up_torch = torch.matmul(hidden_states_torch, up_proj_weight_torch)
    up_activated_torch = gelu_torch(up_torch.float()).to(dtype)
    output_torch_ref = torch.matmul(up_activated_torch, down_proj_weight_torch)
    output = torch.empty(batch_size, hidden_size, dtype=dtype, device=device_local)
    if config.use_dynamic_shape and config.activation == "gelu":
        dynamic_gelu_activation_core(hidden_states_torch, up_proj_weight_torch,
                                     down_proj_weight_torch, output, config)
    else:
        ffn_activation_kernel(hidden_states_torch, gate_proj_weight_torch, up_proj_weight_torch,
                               down_proj_weight_torch, output, config)
    print(f"Dynamic FFN input shape: {hidden_states_torch.shape}")
    print(f"Dynamic FFN output shape: {output_torch_ref.shape}")
    print(f"Dynamic batch size: {batch_size}")
    print(f"Basic batch size: {basic_batch}")
    if RUN_MODE == pypto.RunMode.NPU:
        assert_allclose(output.cpu().to(torch.float32), output_torch_ref.cpu().to(torch.float32), rtol=3e-3, atol=3e-3)


动态 FFN 的写法和前面的 dynamic shape 章节是一致的：只不过这里的分块目标从单个算子变成了整个模块。理解这一点很重要，因为后面的 attention 也会沿用同样的思路。

| 代码 | 含义 |
| --- | --- |
| `basic_batch = 16` | 每次处理 16 个样本，作为动态切块的基本单位。 |
| `num_iterations = ceil_div(batch_size, basic_batch)` | 计算需要循环多少次，保证最后一个尾块也能覆盖。 |
| `pypto.loop(...)` | 在 kernel 里显式描述 batch 循环。 |
| `pypto.view(hidden_states, ...)` | 从大输入中取出当前 batch 块，形成局部视图。 |
| `valid_shape=[(batch_size - batch_offset).min(basic_batch), hidden_size]` | 告诉 PyPTO 当前块可能是尾块，不一定满 `basic_batch`；这里要使用 PyPTO 符号值的 `.min(...)`，不能用 Python 内置 `min(...)`。 |
| `pypto.matmul(hidden_chunk, up_proj_weight, ...)` | 在局部块上做上投影。 |
| `pypto.assemble(output_chunk, [batch_offset, 0], output)` | 把局部结果写回全局输出的对应位置。 |

这种写法的价值在于，它把“动态 batch 的边界管理”和“模块内部的计算逻辑”分开了。外层负责切块和写回，内层只关心当前块怎么做 matmul 和激活。

In [ ]:
test_ffn_dynamic_gelu()


## 6. API 速览

| API | 作用 |
| --- | --- |
| `pypto.sum` | LayerNorm / RMSNorm 里的规约计算 |
| `pypto.sqrt` | 归一化中的方差或均方根开根号 |
| `pypto.cast` | 在 FFN 中做 FP16/FP32 间转换 |
| `pypto.matmul` | FFN 的投影层核心 |
| `pypto.maximum` | ReLU 分支的实现 |
| `pypto.view` | 动态 batch 切块 |
| `pypto.loop` | 动态 batch 的迭代组织 |
| `pypto.assemble` | 将局部结果写回输出 |

如果把这一节压缩成一句话，那就是：归一化负责稳定数值，FFN 负责完成主干映射。

## 7. 课后练习

本节练习用于复盘归一化和 FFN 的组合方式。完成题目时，建议回到 LayerNorm、RMSNorm、静态 FFN 和动态 FFN 的代码中定位对应语句。

1. （选择题）LayerNorm 和 RMSNorm 的主要区别是什么？  
   A. LayerNorm 减均值并按方差归一化，RMSNorm 只按均方根归一化  
   B. RMSNorm 一定需要 beta，LayerNorm 一定不需要 beta  
   C. LayerNorm 不涉及 reduction  
   D. RMSNorm 会改变 batch size
2. （选择题）为什么 LayerNorm 和 RMSNorm 都能用 `sum` 和 `sqrt` 写出来？  
   A. 因为它们都需要沿最后一维做规约，并把结果广播回原 shape  
   B. 因为它们只做矩阵乘法  
   C. 因为它们不需要中间 Tensor  
   D. 因为它们只支持标量输入
3. （填空题）FFN 的三个静态分支里，________ 分支多了一路 gate 投影。
4. （选择题）动态 FFN 为什么要先 `view` 再 `assemble`？  
   A. 为了按固定 tile 处理动态 batch，并把局部结果写回完整输出  
   B. 为了删除 hidden 维度  
   C. 为了把 dtype 改成 INT64  
   D. 为了跳过 down projection
5. （填空题）`config.activation` 在 `ffn_activation_kernel` 中用于________。

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/04.03_answer.txt


## 8. 小结

到这里，你已经把归一化和 FFN 两类中级模块串了起来。它们看起来比前面的算子组合更大，但本质仍然是同一件事：把基础算子按正确的数学结构组织成一个稳定、可验证的 PyPTO kernel。

下一节会继续往前走，进入 dynamic shape、loop 和 condition。